# Day 6 | ILT 3: Fact Tables Deep Dive
### GlobalMart Data Engineering Bootcamp
---
**Duration:** 60 min &nbsp;|&nbsp; **Level:** Intermediate &nbsp;|&nbsp; **Tags:** fact-tables, additive-measures, degenerate-dimensions, natural-keys, fact_sales

---
**Session Time:** Final ILT of Day 6, right before this afternoon's Hands-On
**Goal:** Understand what TYPE of fact table `fact_sales` is, what KIND of measures it holds, and how it references its 6 dimensions — the last conceptual piece before building the star schema this afternoon.

---
**INSTRUCTOR NOTE:**
Grain (ILT2) and dimension design (ILT2) are already settled. This session turns to the fact table itself — its type, its measures, and how it points at its dimensions.

The most important correction to make stick today: **only 2 of `fact_sales`'s 4 measures are additive.** `Actual_price` and `Discounted_price` are not. Get this right — it's the one place today where a wrong worked example would directly undermine the lesson being taught.

By the end of this session, every student should be able to answer:
- *What TYPE of fact table is `fact_sales` — transactional, periodic snapshot, or accumulating snapshot?*
- *Which of `fact_sales`'s 4 measures can you safely SUM? Which two can't you?*
- *`dim_orders` exists as a real Gold table — so why does `Order_ID` still act like a degenerate dimension inside `fact_sales`?*
- *Why does `fact_sales` reference all 6 dimensions by natural key, not the surrogate key each one already has?*</cell id="ilt3-title-01">


## Learning Objectives

By the end of this session, students will be able to:

1. Classify fact tables into transactional, periodic snapshot, and accumulating snapshot — and identify which one `fact_sales` is
2. Classify measures into additive, semi-additive, and non-additive — and correctly classify `Quantity_purchased`, `Sales_amount`, `Actual_price`, and `Discounted_price`
3. Explain why `Order_ID` still behaves like a degenerate dimension inside `fact_sales`, even though `dim_orders` exists as its own Gold table
4. Explain why `fact_sales` joins all 6 dimensions by natural key, not the surrogate key each dimension already has — and what that choice costs
5. Describe `fact_sales`'s final column shape — without building it (that's Day 7)</cell id="ilt3-objectives-02">


---
## Section 1: Where We Are

**INSTRUCTOR NOTE:**
Quick recap — don't spend more than 5 minutes here. This is a bridge, not a re-teach.

---

Today so far:

| Session | What got settled |
|---|---|
| **ILT 1** | Business process = "a sale." Kimball's 4 steps. Star schema shape. |
| **ILT 2** | Grain = **one row per order line item**. All 6 dimensions designed: `dim_customer`, `dim_product`, `dim_date`, `dim_address`, `dim_payment_method`, `dim_orders`. |
| **ILT 3 (this session)** | The fact table itself — its type, its measures, how it points at its dimensions. |
| **This afternoon's Hands-On** | Build all 6 dimensions for real, in Gold. |
| **Day 7** | Build `fact_sales` itself + verify the illustrative bridge-table pattern. |

Two questions are still open, and they're both about the fact table:

1. **What kind of fact table is `fact_sales`?** (There's more than one kind.)
2. **What kind of measures does it hold?** (There's more than one kind — and getting this wrong produces silently wrong dashboards.)

That's this session.</cell id="ilt3-recap-03">


---
## Section 2: Three Types of Fact Tables

**INSTRUCTOR NOTE:**
Draw the distinction sharply: the difference is about WHEN a row is written and whether it ever changes afterward. Use the GlobalMart examples, not abstract ones.

---

Every fact table falls into one of three shapes:

### 1. Transactional Fact
One row per business EVENT, captured at the moment it happens. Grain = the event itself. A row is inserted once and — under normal business rules — never updated again.

> **`fact_sales` is a transactional fact.** Grain = one row per order line item, inserted when the line item is placed. Once an order line is placed, its quantity/price at time of sale doesn't retroactively change. This also matches the append-only philosophy you've already used all week in Bronze and Silver — transactional facts are naturally append-friendly.

### 2. Periodic Snapshot Fact
One row per entity per FIXED TIME PERIOD (daily, weekly, monthly) — not per event. The same entity gets a brand-new row every period, whether or not anything happened to it.

> GlobalMart example (hypothetical — **not built in this course**, shown only to teach the contrast): a `fact_inventory_daily` table with one row per product per day, holding that day's on-hand stock level. Even if nothing sold, product PRD-00001 still gets a row every single day.

### 3. Accumulating Snapshot Fact
One row per PROCESS INSTANCE (e.g. one order), covering its entire lifecycle. Unlike the other two types, this row gets **updated in place** as the process moves through milestones.

> GlobalMart example (hypothetical — teaching contrast only): an order-fulfillment-lifecycle fact keyed on `OrderID`, with milestone columns `order_placed_date`, `shipped_date`, `expected_delivery_date`, `actual_delivery_date` (straight from `orders.csv`'s `OrderDate` / `ShippingDate` / `ExpectedDeliveryDate` / `ActualDeliveryDate`). The SAME row gets its `shipped_date` filled in later, then its `actual_delivery_date` filled in later still. This is the one fact type that breaks the append-only pattern — it requires UPDATE, not just INSERT.

### Comparison

| | Transactional | Periodic Snapshot | Accumulating Snapshot |
|---|---|---|---|
| **Grain** | One row per event | One row per entity per period | One row per process instance |
| **Row written** | Once, at event time | Repeatedly, every period | Once, then updated repeatedly |
| **Update pattern** | Insert-only (append) | Insert-only (append) | Insert + repeated UPDATE |
| **GlobalMart example** | `fact_sales` ✅ (built this course) | `fact_inventory_daily` (hypothetical) | order fulfillment lifecycle (hypothetical) |
| **Real-world example** | Retail POS transactions, ATM withdrawals | Daily bank account balance, monthly inventory count | Loan application → approval → funding pipeline |

**Why `fact_sales` is transactional, concretely:** its grain (one row per order line item) is an event grain, not a time-period grain or a process-instance grain. It gets INSERTed once when Bronze/Silver capture the order line, and Gold never goes back and edits `Quantity_purchased` or `Discounted_price` on a line item after the fact. That's exactly the append-only shape you've built Bronze and Silver around all week — Gold's fact table keeps the same discipline.</cell id="ilt3-fact-types-04">


---
## Section 3: Additive, Semi-Additive, Non-Additive Measures

**INSTRUCTOR NOTE:**
This is the correction section. An earlier draft of this course's spec said "quantity, unit_price, and line_total are all additive" — that draft also predates `fact_sales`'s real 4-measure shape. **Neither is right**, and this is the session where both get fixed. Say so explicitly to students — it's a good teaching moment about why you always check additivity before writing a SUM().

---

A **measure** is a number in the fact table you aggregate (SUM, AVG, COUNT) to answer business questions. Not every measure aggregates the same way. `fact_sales` carries exactly 4 measures. There are three kinds:

### Additive
Safe to SUM across **any** dimension — product, customer, date, category, region, all of them, in any combination.

- **`Quantity_purchased`** — fully additive. `SUM(Quantity_purchased)` grouped by category, by day, by customer — always correct.
- **`Sales_amount`** (= `Quantity_purchased` × `Discounted_price`, computed once when the fact row is built) — fully additive, and it's `fact_sales`'s primary revenue measure. `SUM(Sales_amount)` grouped by category this month, by customer this year, by payment method — always correct.

### Non-Additive
**Never SUM it.** Summing produces a number with no business meaning. `fact_sales` has **two** of these — both prices.

- **`Actual_price`** and **`Discounted_price`** — both non-additive. Each is a *rate* (price per unit), not a quantity of something. Summing the price of a ₹1,216 phone accessory and a ₹499 t-shirt gives you ₹1,715 — a number that answers no business question anyone has ever asked. You still **store** both (you need `Discounted_price` to compute `Sales_amount`, and `Actual_price` to measure how much discount was actually given), you just never aggregate either with SUM.
  - **Wrong:** `AVG(Discounted_price)` across many different products, to get "average selling price."
  - **Right:** average selling price = `SUM(Sales_amount) / SUM(Quantity_purchased)` — this correctly weights by how much of each product actually sold.
  - The one thing you legitimately *do* aggregate these two into: **discount analysis.** Day 7 HOL 2's real `vw_monthly_category_sales` view runs `ROUND(AVG(Actual_price - Discounted_price), 2) AS avg_discount_given` — that's not "averaging a price," it's averaging the *difference* of two rates, which is itself a rate. Averaging the raw price of unrelated products is still wrong; averaging the discount gap is a meaningful, intentional exception.

### Semi-Additive
Safe to SUM across **most** dimensions, but **not across time**.

> GlobalMart doesn't have a semi-additive measure in `fact_sales` — flagging this as a concept you'll meet in other fact tables. Classic example: **inventory on-hand level** (or a bank account balance).
>
> Picture 3 GlobalMart warehouses, each holding 100 units of PRD-00001 today. Summing on-hand *across warehouses* today: 100 + 100 + 100 = 300 units on-hand right now. Valid.
>
> Now picture one warehouse's on-hand level over 7 days: 100, 100, 100, 100, 100, 100, 100 (nothing sold or restocked all week). Summing on-hand *across those 7 days* gives you 700 — which is nonsense. It's still the same 100 units sitting on the same shelf; you've counted the same stock seven times. The fix for the time dimension is to **average** it or take the **period-end value**, never SUM across days.

### Summary Table

| Measure | Additivity | Safe aggregation |
|---|---|---|
| `Quantity_purchased` | Additive | `SUM(Quantity_purchased)` across any dimension |
| `Sales_amount` | Additive | `SUM(Sales_amount)` across any dimension — this is "revenue" |
| `Actual_price` | **Non-additive** | Never SUM/AVG across products directly; combine with `Discounted_price` for discount analysis |
| `Discounted_price` | **Non-additive** | Same rule; derive avg. selling price as `SUM(Sales_amount)/SUM(Quantity_purchased)` instead |
| *(inventory on-hand — not in fact_sales)* | Semi-additive | SUM across product/warehouse OK; SUM across time is wrong — average or use period-end value instead |

**Rule of thumb:** before you write `SUM()` on any measure, ask "does adding these two numbers together produce something a business person would recognize as meaningful?" If the answer is no, it's not additive on that dimension.</cell id="ilt3-measures-05">


---
## Section 4: Degenerate Dimensions — and GlobalMart's Twist on the Textbook Case

**INSTRUCTOR NOTE:**
This section has a genuine plot twist — use it. The textbook definition says a degenerate dimension has *no* backing table. GlobalMart built the backing table anyway (`dim_orders`, this afternoon's Hands-On) and the fact table *still* doesn't join to it by surrogate key. Walk students through why both things are true at once — that's the actual lesson, not the textbook definition on its own.

---

A **degenerate dimension** is, by the textbook definition, an identifier that lives in the fact table with **no corresponding dimension table of its own** — usually because it has no descriptive attributes beyond the ID itself.

> By that definition, `Order_ID` looks exactly like a classic degenerate dimension candidate: `orders.csv`'s header-level columns (`OrderDate`, `ShippingTierID`, `SupplierID`, `OrderChannel`) don't look rich enough on their own to justify a whole table.

**But GlobalMart built `dim_orders` anyway** — this afternoon's Hands-On constructs it from `<your-catalog>.silver.orders`, complete with its own surrogate key (`order_sk = sha2(order_id, 256)`) and `order_channel`/`shipping_tier_id`/`supplier_id`/`order_date`. So which is it — degenerate, or a real dimension?

### Both, depending which table you're looking from

| Looking from... | What `Order_ID` is |
|---|---|
| `dim_orders` itself | A real, standalone Gold dimension — one row per order, its own surrogate key, independently queryable ("orders by channel," "orders by supplier") without ever touching `fact_sales` |
| `fact_sales` | `Order_ID` sits in the fact as a **plain natural-key column** — `fact_sales` does **not** join through `dim_orders.order_sk` at all (see Section 5) |

From `fact_sales`'s point of view, `Order_ID` still *does the job* a degenerate dimension does, even though a real dimension exists elsewhere:

1. **Distinguishing orders from line items.** `COUNT(*)` on `fact_sales` counts line items. `COUNT(DISTINCT Order_ID)` counts orders — Day 7 HOL 2's real `vw_regional_sales` view uses exactly this pattern. Both numbers matter and they're different.
2. **Grouping line items back into their parent order.** Order `OR-094494` might have 2 line items (`PRD-00410` qty 1, `PRD-00107` qty 5) — `Order_ID` is what ties those 2 fact rows back together as "one shopping cart," with no need to touch `dim_orders` for that.
3. **Drill-through.** An analyst looking at a suspicious line item can jump straight back to the source order record using `Order_ID` — and *from there*, optionally, to `dim_orders` for the order-header attributes.

### The actual lesson

Having a real dimension table for something doesn't force every fact table to join to it by surrogate key. GlobalMart's `fact_sales` treats `Order_ID` as if it were degenerate — natural key, no `order_sk` join — purely because Section 5's natural-key simplification applies uniformly to *all six* dimensions, `dim_orders` included. `dim_orders` isn't wasted: it just serves order-level analysis directly, on its own, rather than through the fact table.</cell id="ilt3-degenerate-06">


---
## Section 5: Why `fact_sales` Uses Natural Keys, Not the Surrogate Keys Its Dimensions Already Have

**INSTRUCTOR NOTE:**
ILT2 Section 7 already introduced this shortcut while designing the dimensions. This is the payoff session: sitting on the fact-table side, looking at the real cost. Don't re-derive it from scratch — point back to ILT2 and build on it.

---

Every one of GlobalMart's 6 dimensions has a proper surrogate key: `customer_sk`/`product_sk` (reused from Silver's SCD2 build), `address_sk`/`payment_method_sk`/`order_sk` (generated fresh in Gold via `sha2(natural_key, 256)`), and `dim_date.date_key` (the `YYYYMMDD` integer exception). The Kimball-standard rule — and cert material — says a fact table should join to its dimensions on exactly those surrogate keys, never the natural key.

**`fact_sales` does not follow that rule.** It carries `Customer_ID`, `Product_ID`, `Order_ID`, `Address_ID`, `Payment_ID`, and `Time_ID` — six natural/business keys. **Five** of them join their dimension directly: `Customer_ID`→`dim_customer`, `Product_ID`→`dim_product`, `Order_ID`→`dim_orders`, `Address_ID`→`dim_address`, `Time_ID`→`dim_date` (`Time_ID` *is* literally `dim_date.date_key`, since natural and surrogate collapse into the same integer for that one dimension). **`Payment_ID` is the odd one out** — it's a natural key, but it doesn't resolve to `dim_payment_method` at all, because the two live in different ID spaces (Section 4 covers this in full). The only surrogate key `fact_sales` actually generates is its own row identity, `fact_sales_sk = sha2(order_item_id, 256)` — a key for the fact row itself, not a foreign key to anything.

### Why this is a real cost, not a free simplification

The Kimball rule exists specifically to protect **point-in-time accuracy** under SCD Type 2. Picture a customer whose email changes today — `dim_customer` (Type 2) keeps both rows: the old one marked `is_current = false`, the new one `is_current = true`, each with its own `customer_sk`. If `fact_sales` joined on `customer_sk`, every past sale would still resolve to the exact dimension row version that was true *when that sale happened*. Because `fact_sales` instead joins on the bare `Customer_ID`, every join today — including joins against last year's sales — resolves to whichever `dim_customer` row currently has that `Customer_ID`. A `Customer_ID` shared across two SCD2 versions doesn't tell the join which version was true at sale time; it just picks up today's.

### Why GlobalMart still made this choice

1. **Uniformity.** The five dimension-joining keys get exactly the same natural-key treatment from the fact table — no special-case join logic to remember per dimension. ILT2's `dim_orders` section (Section 6) already flagged this: even the newest, most obviously dimension-shaped table still doesn't get a surrogate-key join from the fact.
2. **Simplicity for a training build.** Point-in-time accuracy matters most for `dim_customer`/`dim_product` (the two that actually carry SCD2 history) — and even for those two, GlobalMart's real fact table build takes the simpler path.
3. **It's documented, not hidden.** Day 7 HOL 2 states this explicitly as one of its Key Takeaways: *"Natural-key joins were a deliberate choice, not a shortcut — you now know exactly what it costs (no point-in-time accuracy) and why it was still the right call for this program."*

> **Rule to internalize regardless:** the Kimball-standard, cert-tested default is still *join facts to dimensions on the surrogate key*. GlobalMart's `fact_sales` is the documented exception you now understand the cost of — not the pattern to reach for by default in your own designs.</cell id="ilt3-surrogate-07">


---
## Section 6: Putting It Together — `fact_sales`'s Column Shape

**INSTRUCTOR NOTE:**
This is a sketch, not a build. Say explicitly: "We are not writing a single line of code for `fact_sales` today." That's Day 7.

If a sharp student asks "does `Payment_ID` join to `dim_payment_method`?" — the honest answer is no. `Payment_ID` comes from `silver.payments` (the transactional payment record, one per order) and is a completely different ID space from `dim_payment_method.payment_method_id` (the ~5-row method lookup, "UPI"/"Credit Card"). `fact_sales` carries `Payment_ID` the same degenerate-style way it carries `Order_ID` — it does not resolve to `dim_payment_method` through the fact table at all.

---

Here's what every row of `fact_sales` will look like once it's built (Day 7):

| Column | Role | Type | Notes |
|---|---|---|---|
| `fact_sales_sk` | Surrogate PK — the only generated key in this table | hash/string | `sha2(order_item_id, 256)` — hashed from the grain's own natural key |
| `Payment_ID` | Natural key, from `silver.payments` | string | 1:1 with the order; degenerate-style — not the same ID as `dim_payment_method.payment_method_id` |
| `Customer_ID` | Natural key → `dim_customer` | string | No `customer_sk` join — see Section 5 |
| `Product_ID` | Natural key → `dim_product` | string | Looked up filtering `is_current = true` |
| `Order_ID` | Natural key → `dim_orders` | string | Also functions as the fact's degenerate-dimension-style grouping/drill-through key — see Section 4 |
| `Address_ID` | Natural key → `dim_address` | string | Resolved via the one-primary-address-per-customer ranking (ILT 1) |
| `Time_ID` | Natural key → `dim_date` | int (YYYYMMDD) | This one *is* `dim_date.date_key` — natural and surrogate collapse into the same column |
| `Quantity_purchased` | Measure — additive | int | |
| `Actual_price` | Measure — **non-additive** | decimal | Store it, never SUM it |
| `Discounted_price` | Measure — **non-additive** | decimal | What the customer actually paid per unit |
| `Sales_amount` | Measure — additive | decimal | `Quantity_purchased` × `Discounted_price` — the revenue number |

Five dimension-joining natural keys, one degenerate-style natural key (`Payment_ID`), four measures, one fact-owned surrogate key. Nothing here gets built until Day 7 — **today's hands-on only builds the six dimension tables this fact will eventually point to** (`dim_payment_method` included — it's still a real, independently useful Gold table, the same story as `dim_orders` in Section 4).</cell id="ilt3-column-shape-08">


---
## Session Summary

| Topic | Key Takeaway |
|---|---|
| **Fact table type** | `fact_sales` is a **transactional fact** — one row per order line item, insert-only |
| **Additive measures** | `Quantity_purchased` and `Sales_amount` — safe to SUM across any dimension |
| **Non-additive measures** | `Actual_price` and `Discounted_price` — never SUM/AVG across products; derive avg. selling price as `SUM(Sales_amount)/SUM(Quantity_purchased)` instead |
| **Semi-additive (not in fact_sales)** | Inventory on-hand / account balance — SUM across entities OK, SUM across time is wrong |
| **Order_ID's dual role** | Joins `dim_orders` directly on the natural key (a real, genuine dimension) **and** doubles as the fact's degenerate-style grouping/drill-through key, since that join never goes through `dim_orders.order_sk` — Section 4 |
| **Payment_ID — the pure non-joiner** | Unlike `Order_ID`, `Payment_ID` doesn't resolve to `dim_payment_method` through any column at all — different ID space entirely |
| **Natural vs. surrogate keys** | All 6 dimensions have proper surrogate keys — `fact_sales` still joins on natural keys anyway, at the cost of point-in-time SCD2 accuracy |
| **What's NOT built today** | `fact_sales` itself — that's Day 7 |

---

### What's Next

| Session | Topic |
|---|---|
| **This afternoon — Hands-On** | Star Schema Design — build all 6 dimension tables (`dim_customer`, `dim_product`, `dim_date`, `dim_address`, `dim_payment_method`, `dim_orders`) for real, in Gold |
| **Day 7** | Build Gold Layer — `fact_sales` itself, plus a verification pass and an illustrative bridge-table example |

---

**INSTRUCTOR NOTE:**
Closing activity (5 minutes): Ask each student to answer one question out loud:
1. *'Name `fact_sales`'s two non-additive measures.'* (`Actual_price`, `Discounted_price`)
2. *'`dim_orders` exists as a real table — so why does `Order_ID` still act like a degenerate dimension inside `fact_sales`?'* (it's carried as a natural key, not joined via `order_sk` — even though the join itself would work fine on the natural key if you wrote it)
3. *'Which of the six natural keys doesn't resolve to its "matching" dimension at all, and why?'* (`Payment_ID` — it's from `silver.payments`, a different ID space than `dim_payment_method.payment_method_id`)
4. *'What type of fact table is `fact_sales` — transactional, periodic snapshot, or accumulating snapshot?'* (transactional)</cell id="ilt3-summary-09">
